# Voice Model Comparison

Run from this directory. Warmups excluded; all measured trials retained. This single cell was executed with Python stdlib (nbclient unavailable). See the HTML report for findings and caveats.

In [1]:
import json, statistics, math
from collections import Counter
from pathlib import Path
rows = [json.loads(l) for l in Path("voice-model-comparison-20260909.jsonl").read_text().splitlines()]
trials = [r for r in rows if not r["warmup"]]
assert len(rows) == 74 and len(trials) == 72
assert len({(r["case"],r["repeat"],r["model"]) for r in trials}) == 72
summary = []
for model in sorted({r["model"] for r in trials}):
    group = [r for r in trials if r["model"] == model]
    assert len(group) == 36
    assert set(Counter(r["case"] for r in group).values()) == {3}
    times = sorted(r["ms"] for r in group)
    summary.append(dict(model=model, n=36, median_s=statistics.median(times)/1000,
        p95_s=times[math.ceil(.95*len(times))-1]/1000, max_s=max(times)/1000,
        median_chars=statistics.median(len(r.get("reply","")) for r in group),
        errors=sum("error" in r or "generation_fallback" in r["issues"] for r in group)))
pairs = {}
for r in trials:
    pairs.setdefault((r["case"],r["repeat"]), []).append(r)
diffs = []
for pair in pairs.values():
    a,b = sorted(pair,key=lambda r:r["model"])
    assert a["student"] == b["student"] and a["previous_reply"] == b["previous_reply"]
    diffs.append(b["ms"]-a["ms"])
paired = dict(n=len(diffs), larger_slower=sum(d>0 for d in diffs), median_delta_ms=statistics.median(diffs))
print(json.dumps(dict(summary=summary, paired=paired),ensure_ascii=False,indent=2))


{
  "summary": [
    {
      "model": "Qwen/Qwen3.5-9B",
      "n": 36,
      "median_s": 3.054,
      "p95_s": 4.168,
      "max_s": 4.296,
      "median_chars": 90.0,
      "errors": 0
    },
    {
      "model": "Qwen/Qwen3.8-27B",
      "n": 36,
      "median_s": 4.583,
      "p95_s": 5.484,
      "max_s": 5.752,
      "median_chars": 85.5,
      "errors": 0
    }
  ],
  "paired": {
    "n": 36,
    "larger_slower": 31,
    "median_delta_ms": 1480.0
  }
}
